# 05b — DenseNet-121: training dari awal (exp_005b_densenet121_scratch)

**Pendekatan 1:** DenseNet-121 dimulai dari bobot acak, lalu seluruh lapisannya
dilatih pada dataset awan **GCD + CCSN**, dengan keluaran **7 kelas**.
Konfigurasi `pretrained: false` diteruskan menjadi `weights=None`;
`finetune: full` membuat backbone dan classifier sama-sama dilatih.
Tidak ada bobot ImageNet atau checkpoint sebelumnya yang dimuat saat inisialisasi.

Notebook memakai modul proyek di `src/` dan konfigurasi khusus
`configs/experiments/exp_005b_densenet121_scratch.yaml`.
Jalankan sel berurutan. Hasil dan checkpoint memakai nama eksperimen tersebut.
Lihat `docs/pelatihan/TRAINING_GUIDE.md` untuk aturan split dan evaluasi.


## 1. Setup & cek perangkat

In [14]:
import os, sys
from pathlib import Path

# Pastikan berjalan dari akar repo (folder yang memuat src/).
CURRENT = Path.cwd().resolve()

REPO = next(
    (
        folder
        for folder in (CURRENT, *CURRENT.parents)
        if (folder / "src").is_dir()
    ),
    None,
)

if REPO is None:
    raise FileNotFoundError(
        f"Folder 'src' tidak ditemukan dari {CURRENT} "
        "maupun folder induknya. Periksa lokasi proyek."
    )

os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
print("repo   :", REPO)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available(),
      "(" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "(CPU)")

repo   : D:\n8n-logsiswaparalayang\Jurnal\CitraAwan
torch  : 2.1.1+cu121
cuda   : True (NVIDIA GeForce RTX 4060 Laptop GPU)


## 2. Konfigurasi run

- `EPOCHS=None` memakai konfigurasi eksperimen: **10 epoch**, sesuai pengaturan
  awal notebook ini. Gunakan `EPOCHS=1` untuk uji cepat.
- `BATCH_SIZE=None` memakai konfigurasi: **8**. Pelatihan seluruh backbone
  memerlukan memori lebih banyak daripada hanya melatih classifier.
  Jika kehabisan memori, turunkan menjadi 4 atau 2.
- `NUM_WORKERS=0` mengikuti konfigurasi agar mudah dijalankan di Jupyter Windows.
- Input tetap **512×512 RGB**, normalisasi **/255** ke [0, 1].
- Optimizer **AdamW**, learning rate **1e-4**, mengikuti konfigurasi dasar.
  Sepuluh epoch adalah pengaturan awal, bukan jaminan konvergensi dari bobot acak.
  Penyesuaian dilakukan berdasarkan validation set.

Konfigurasi efektif disimpan sebelum pelatihan. Pemilihan checkpoint memakai
**validation macro-F1**; evaluasi test tersedia pada sel terpisah di akhir.


In [15]:
from src.utils.config import load_config, set_seed, resolve_device

CONFIG_PATH = "configs/experiments/exp_005b_densenet121_scratch.yaml"
EPOCHS = None       # None = config (10); 1 untuk uji cepat
BATCH_SIZE = None   # None = config (8); turunkan ke 4 atau 2 bila OOM
NUM_WORKERS = None # None = config (0)
DEVICE = "auto"    # "auto" | "cpu" | "cuda"

cfg = load_config(CONFIG_PATH)
for key, value in (
    ("epochs", EPOCHS),
    ("batch_size", BATCH_SIZE),
    ("num_workers", NUM_WORKERS),
):
    if value is not None:
        cfg["training"][key] = value

# Cegah notebook training dari awal memakai bobot pretrained/backbone beku.
assert cfg["model"]["architecture"] == "densenet121"
assert cfg["model"]["pretrained"] is False, "Training dari awal memerlukan pretrained: false."
assert cfg["model"]["finetune"] == "full", "Seluruh parameter harus dilatih."
assert cfg["model"]["num_classes"] == cfg["data"]["num_classes"]
set_seed(cfg["seed"])
device = resolve_device(DEVICE)
name = cfg["name"]
print(f"eksperimen {name} | device {device} | epochs {cfg['training']['epochs']} "
      f"| batch {cfg['training']['batch_size']} | workers {cfg['training']['num_workers']}")


eksperimen exp_005b_densenet121_scratch | device cuda | epochs 10 | batch 8 | workers 0


## 3. Data (train seimbang 21.000 / validation 1.851)

Train menggunakan `combined_balanced` (3.000 citra per kelas).
Validation memakai split gabungan yang sudah tersedia.
Test resmi 9.000 citra hanya dimuat saat evaluasi akhir.


In [16]:
from src.datasets.cloud_dataset import build_dataloader

train_loader = build_dataloader("train", cfg)
val_loader = build_dataloader("val", cfg)
print("sampel -> train:", len(train_loader.dataset), "| val:", len(val_loader.dataset))
print("batches -> train:", len(train_loader), "| val:", len(val_loader))


sampel -> train: 21000 | val: 1851
batches -> train: 2625 | val: 232


## 4. Model: bobot acak, seluruh parameter dilatih

In [17]:
from src.models.baseline import build_model, count_parameters

# Pasang ulang seed agar inisialisasi dapat diulang saat sel ini dijalankan ulang.
set_seed(cfg["seed"])
model = build_model(
    cfg["model"]["architecture"],
    num_classes=cfg["data"]["num_classes"],
    pretrained=cfg["model"]["pretrained"],  # False -> torchvision weights=None
    finetune=cfg["model"]["finetune"],      # full -> semua parameter dilatih
)
total, trainable = count_parameters(model)
assert total == trainable, "Masih ada parameter model yang dibekukan."
print(f"DenseNet-121 | bobot acak | parameter total {total:,} | trainable {trainable:,}")
print("classifier:", model.classifier)


DenseNet-121 | bobot acak | parameter total 6,961,031 | trainable 6,961,031
classifier: Linear(in_features=1024, out_features=7, bias=True)


## 5. Pelatihan dari awal

Sel ini menjalankan pelatihan sesuai jumlah epoch di konfigurasi.
Checkpoint terbaik disimpan ke `checkpoints/exp_005b_densenet121_scratch/best.pt`
berdasarkan validation macro-F1.

Untuk mengulang pelatihan **dari awal**, jalankan kembali sel model terlebih
dahulu. Menjalankan ulang sel pelatihan saja memakai bobot model yang masih
berada di memori.


In [18]:
import json
import yaml
from dataclasses import asdict
import pandas as pd
from src.training.trainer import fit

exp_dir = Path(cfg["output"]["experiments_dir"]) / name
exp_dir.mkdir(parents=True, exist_ok=True)
(exp_dir / "config.yaml").write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8"
)
(exp_dir / "model_architecture.txt").write_text(
    f"{model}\n\nTotal parameters: {total}\nTrainable parameters: {trainable}\n",
    encoding="utf-8",
)
ckpt = Path(cfg["checkpoint"]["dir"]) / name / "best.pt"
result = fit(model, train_loader, val_loader, cfg, device, ckpt, progress=True)

hist = pd.DataFrame([asdict(record) for record in result.history])
hist.to_csv(exp_dir / "training_history.csv", index=False)
(exp_dir / "metrics.json").write_text(
    json.dumps({
        "experiment": name,
        "best_epoch": result.best_epoch,
        "monitor": cfg["checkpoint"]["monitor"],
        "best_val_metric": result.best_metric,
        "test": {"status": "not_evaluated"},
    }, indent=2),
    encoding="utf-8",
)
print(f"SELESAI. best_epoch={result.best_epoch} | best val_macro_f1={result.best_metric:.4f}")
print("checkpoint:", ckpt)
print("artefak:", exp_dir)


Epoch: 1/10
2625/2625 [================] - 1958s 746ms/step - loss: 1.1052 - train_loss: 1.1055 - val_loss: 0.8367 - val_macro_f1: 0.6439 - val_bal_acc: 0.6339 - val_acc: 0.6829
   ^ best sejauh ini (epoch 1, val_macro_f1 0.6439)
Epoch: 2/10
2625/2625 [================] - 1941s 740ms/step - loss: 0.8550 - train_loss: 0.8552 - val_loss: 0.7293 - val_macro_f1: 0.6904 - val_bal_acc: 0.6815 - val_acc: 0.7169
   ^ best sejauh ini (epoch 2, val_macro_f1 0.6904)
Epoch: 3/10
2625/2625 [================] - 1496s 570ms/step - loss: 0.7526 - train_loss: 0.7524 - val_loss: 0.7329 - val_macro_f1: 0.6762 - val_bal_acc: 0.6524 - val_acc: 0.7158
Epoch: 4/10
2625/2625 [================] - 1275s 486ms/step - loss: 0.6894 - train_loss: 0.6894 - val_loss: 0.7344 - val_macro_f1: 0.7148 - val_bal_acc: 0.7331 - val_acc: 0.7304
   ^ best sejauh ini (epoch 4, val_macro_f1 0.7148)
Epoch: 5/10
2625/2625 [================] - 1241s 473ms/step - loss: 0.6356 - train_loss: 0.6358 - val_loss: 0.6119 - val_macro_f1: 0

## 6. Ringkasan riwayat pelatihan

In [19]:
from IPython.display import display

display(hist.round(4))
history_plot = hist.set_index("epoch")
history_plot[["train_loss", "val_loss"]].plot(
    title=f"{name}: loss per epoch", figsize=(7, 4)
).set_xlabel("epoch")
history_plot[["val_macro_f1", "val_balanced_accuracy", "val_accuracy"]].plot(
    title=f"{name}: metrik validation per epoch", figsize=(7, 4)
).set_xlabel("epoch")


,epoch,train_loss,val_loss,val_macro_f1,val_balanced_accuracy,val_accuracy
0,1,1.1055,0.8367,0.6439,0.6339,0.6829
1,2,0.8552,0.7293,0.6904,0.6815,0.7169
2,3,0.7524,0.7329,0.6762,0.6524,0.7158
3,4,0.6894,0.7344,0.7148,0.7331,0.7304
4,5,0.6358,0.6119,0.7623,0.7858,0.7688
5,6,0.5925,0.6659,0.7346,0.7256,0.7472
6,7,0.5593,0.5958,0.7647,0.7574,0.7720
7,8,0.5199,0.5812,0.7722,0.7636,0.7812
8,9,0.4983,0.6946,0.6939,0.6890,0.7515
9,10,0.4738,0.5492,0.7793,0.7728,0.7947


Text(0.5, 0, 'epoch')

## 7. Evaluasi test akhir (opsional, sekali setelah model dipilih)

Ubah `EVALUATE_TEST=True` hanya setelah keputusan model dan hyperparameter
selesai berdasarkan validation. Evaluasi memuat **checkpoint terbaik dari
pelatihan ini**, lalu menyimpan metrik, classification report, dan confusion
matrix ke folder eksperimen.


In [21]:
EVALUATE_TEST = True

if EVALUATE_TEST:
    from src.training.trainer import build_criterion, evaluate
    from src.evaluation.metrics import compute_metrics
    from scripts.train import write_outputs

    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state["model_state_dict"])
    model.to(device)
    test_loader = build_dataloader("test", cfg)
    _, y_true, y_pred = evaluate(model, test_loader, build_criterion(cfg, device), device)
    test_metrics = compute_metrics(y_true, y_pred, cfg["data"]["num_classes"])
    write_outputs(exp_dir, cfg, result, test_metrics, y_true, y_pred)
    print(f"TEST macro-F1 {test_metrics['macro_f1']:.4f} | "
          f"balanced acc {test_metrics['balanced_accuracy']:.4f} | "
          f"acc {test_metrics['accuracy']:.4f}")
else:
    print("Test belum dievaluasi. Gunakan validation untuk pemilihan model.")


TEST macro-F1 0.6583 | balanced acc 0.6442 | acc 0.6983
